In [3]:
import numpy as np
import pdfplumber
import pandas as pd
import re

In [14]:
def cargar_pdf(ruta_pdf: str):
    f = open(ruta_pdf, "rb")
    pdf = pdfplumber.open(f)
    return pdf, f

def extraer_texto_paginas(pdf) -> str:
    texto = ""
    for p in pdf.pages:
        contenido = p.extract_text()
        if contenido:
            texto += contenido + "\n"
    return texto

def limpiar_lineas(texto: str) -> list:
    lineas = [line.strip() for line in texto.split("\n") if line.strip()]
    return lineas

def extraer_transacciones(lineas: list) -> pd.DataFrame:
    patron = re.compile(
        r"(\d{2}/\d{2})\s+(.*?)\s+(\d{1,3}(?:\.\d{3})*,\d{2})"
    )

    registros = []
    for linea in lineas:
        match = patron.search(linea)
        if match:
            fecha, establecimiento, valor = match.groups()
            registros.append({
                "fecha": fecha,
                "establecimiento": establecimiento,
                "valor": float(valor.replace(".", "").replace(",", "."))
            })

    return pd.DataFrame(registros)

def cargar_estado_cuenta(ruta_pdf: str) -> pd.DataFrame:
    pdf, f = cargar_pdf(ruta_pdf)
    texto = extraer_texto_paginas(pdf)
    lineas = limpiar_lineas(texto)
    df = extraer_transacciones(lineas)
    pdf.close()
    f.close()
    return df

ruta = "D:/DRVACAE/python_projects/presupuesto/data/diners.pdf"

df = cargar_estado_cuenta(ruta)
print(df)
df.to_excel("output/estado_cuenta.xlsx", index=False)

    fecha                      establecimiento   valor
0   06/02  28 CASH ADVANCE PREFERENTE 2 (7/60)  300.31
1   22/07    6797970 PLAN DEUDA ASEGURADA PLUS    8.66
2   16/08             3276298 DN DTV*DIRECTVGO   19.99
3   16/08    3276298 RET IVA SERV DIGITAL 100%    3.00
4   23/07                2646070 AE Jack Jones   82.35
5   18/07         2543750 63 Dlocal*UBER RIDES    2.97
6   18/07    2543750 RET IVA SERV DIGITAL 100%    0.45
7   16/08         3311592 63 Dlocal*UBER RIDES    6.90
8   16/08    3311592 RET IVA SERV DIGITAL 100%    1.04
9   22/07  2662872 018 MUeLLER KARLSRUHE 2-9 E   82.71
10  18/08              IMPUESTO SALIDA DIVISAS    9.71
11  03/08       15478 SU PAGO "MUCHAS GRACIAS"  374.74
12  16/08                     DN DTV*DIRECTVGO    0.00
13  23/07                    AE Jack Jones EUR   69.98
14  18/07                 63 Dlocal*UBER RIDES    0.00
15  16/08                 63 Dlocal*UBER RIDES    0.00
16  22/07      018 MUeLLER KARLSRUHE 2-9 E EUR   70.29
